# TSL Tutorial

- TSL documentation: https://torch-spatiotemporal.readthedocs.io/en/latest/
- TSL github: https://github.com/TorchSpatiotemporal/tsl

In [6]:
import tsl
import torch
import numpy as np
import pandas as pd

print(f"tsl version  : {tsl.__version__}")
print(f"torch version: {torch.__version__}")

# Utility functions ################
def print_matrix(matrix):
    return pd.DataFrame(matrix)

def print_model_size(model):
    tot = sum([p.numel() for p in model.parameters() if p.requires_grad])
    out = f"Number of model ({model.__class__.__name__}) parameters:{tot:10d}"
    print("=" * len(out))
    print(out)

tsl version  : 0.9.5
torch version: 2.2.0


In [7]:
from tsl.datasets import MetrLA

In [26]:
dataset = MetrLA(root='./data')
print(dataset)

MetrLA(length=34272, n_nodes=207, n_channels=1)


/opt/anaconda3/envs/tsl/lib/python3.10/site-packages/tsl/datasets/metr_la.py:98: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(df.index[0], df.index[-1], freq='5T')
/opt/anaconda3/envs/tsl/lib/python3.10/site-packages/tsl/datasets/metr_la.py:109: FutureWarning: The 'method' keyword in DataFrame.replace is deprecated and will be removed in a future version.
  df = df.replace(to_replace=0., method='ffill')


In [11]:
print(f"Sampling period: {dataset.freq}")
print(f"Has missing values: {dataset.has_mask}")
print(f"Percentage of missing values: {(1 - dataset.mask.mean()) * 100:.2f}%")
print(f"Has exogenous variables: {dataset.has_covariates}")
print(f"Covariates: {', '.join(dataset.covariates.keys())}")

Sampling period: <5 * Minutes>
Has missing values: True
Percentage of missing values: 8.11%
Has exogenous variables: True
Covariates: dist


In [12]:
print_matrix(dataset.dist)

,0,1,2,3,4,5,6,7,8,9,...,197,198,199,200,201,202,203,204,205,206
0,0.000000e+00,inf,inf,inf,inf,inf,inf,inf,inf,inf,...,inf,inf,3.764700e+03,inf,inf,9204.299805,inf,inf,8114.799805,1.000970e+04
1,inf,0.000000e+00,2.504600e+03,8563.799805,8572.500000,9561.000000,9590.000000,2.506300e+03,inf,inf,...,inf,inf,inf,inf,4941.899902,7559.200195,7877.200195,inf,inf,inf
2,inf,1.489300e+03,0.000000e+00,6971.299805,6978.299805,9148.200195,9177.099609,3.995700e+03,inf,inf,...,inf,9467.799805,inf,inf,6431.399902,7821.799805,9366.599609,inf,inf,9.837000e+03
3,inf,6.805900e+03,9.293600e+03,0.000000,1745.500000,6068.799805,5401.500000,9.312300e+03,inf,inf,...,inf,5906.500000,inf,inf,inf,inf,inf,inf,inf,7.604400e+03
4,inf,6.606700e+03,9.111300e+03,1767.400024,0.000000,4464.000000,3655.899902,9.113100e+03,inf,inf,...,inf,7207.100098,inf,inf,inf,inf,inf,inf,inf,8.905000e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,inf,inf,1.098710e+04,inf,inf,inf,inf,1.013360e+04,inf,inf,...,inf,inf,inf,inf,9098.900391,0.000000,4072.699951,inf,inf,inf
203,inf,1.072320e+04,9.461800e+03,inf,inf,inf,inf,8.608300e+03,inf,inf,...,inf,inf,inf,inf,7283.500000,inf,0.000000,inf,inf,inf
204,inf,inf,inf,inf,inf,inf,inf,inf,9189.799805,3.171100e+03,...,3672.399902,inf,inf,inf,inf,inf,inf,0.0,inf,inf
205,9.599800e+03,inf,inf,inf,inf,inf,inf,inf,inf,1.016750e+04,...,inf,inf,1.050080e+04,inf,inf,inf,inf,inf,0.000000,inf


In [13]:
connectivity = dataset.get_connectivity(layout='edge_index')

In [17]:
edge_index, edge_weight = connectivity

In [18]:
from tsl.ops.connectivity import edge_index_to_adj

In [21]:
adj = edge_index_to_adj(edge_index, edge_weight)
print(adj.shape)

(207, 207)


In [24]:
from tsl.data import SpatioTemporalDataset

In [29]:
torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=connectivity,
                                      mask=dataset.mask,
                                      horizon=12,
                                      window=12,
                                      stride=1)
print(torch_dataset)

RuntimeError: Could not infer dtype of numpy.float32